In [ ]:
import arcpy
import os
import arcpy.sa


## This creates longest flow path for catchments obtained from MERIT Hydro dataset, such that 
 lfp = MAX(Downstream + Upstream)

In [ ]:

def lfp_merit_hydro(cat_all = r'.\MyProject.gdb\bavaria_catchments',
                    cat_poly_dir = r'.\catchments',
                    fdr_path = r'.\merit_fdr.tif',
                    clip_fdr_save_dir = r'.\lfp_outputs\0merit_hydro\1 clipped fdr',
                    us_dir = r'.\lfp_outputs\0merit_hydro\2a us fl',
                    ds_dir = r'.\lfp_outputs\0merit_hydro\2b ds fl',
                    fl_dir = r'.\lfp_outputs\0merit_hydro\4 total fl',
                    mask_fl_dir = r'.\lfp_outputs\0merit_hydro\5 mask fl',
                    reclass_dir = r'.\lfp_outputs\0merit_hydro\6 reclass fl',
                    lfp_raster_dir = r'.\lfp_outputs\0merit_hydro\7 lfp raster',
                    lfp_polyline_dir = r'.\lfp_outputs\0merit_hydro\8 lfp polyline',
                    clip_lfp_polyl_dir = r'.\lfp_outputs\0merit_hydro\9 clipped lfp'):
    '''
    This is the same method of calculating the flowlength raster as in Lower Saxony.
    Starts from calculating upstream and downstream flowlength raster. 
    Geodetic length is calculated for lfp, as work was done in WGS84 without projecting the raster.
    '''
    with arcpy.da.SearchCursor(cat_all, ['Shape@', 'Pegel_ID']) as cursor:
        for row in cursor:
            # if row[2] == 0: needed when doing with catchments all, and require bavaria only
            pegel_id = int(row[1])
            if pegel_id in [11432002]: #this can be done to work on specific catchments only
                cat_shp = os.path.join(cat_poly_dir, f'original_{pegel_id}.shp')
                fdr_ras = os.path.join(clip_fdr_save_dir, f'fdr_{pegel_id}.tif')
                #1 clip fdr raster
                arcpy.management.Clip(fdr_path, '#',fdr_ras, cat_shp, '#', 'NONE', 'NO_MAINTAIN_EXTENT')
                #2 calculate us and ds fl
                us_ras = os.path.join(us_dir, f'us_{pegel_id}.tif')
                ds_ras = os.path.join(ds_dir, f'ds_{pegel_id}.tif')
                tmp_ds = arcpy.sa.FlowLength(fdr_ras, "DOWNSTREAM", "")
                tmp_ds.save(ds_ras)
                tmp_us = arcpy.sa.FlowLength(fdr_ras, "UPSTREAM", "")
                tmp_us.save(us_ras)
                #4 total fl
                total_ras = os.path.join(fl_dir, f'total_{pegel_id}.tif')
                tmp_fl = arcpy.sa.RasterCalculator([us_ras, ds_ras], ['x', 'y'], '(x*100000)+(y*100000)') #1e5 because wgs84 coordinates at 0.00000raster_width
                tmp_fl.save(total_ras)
                #5 mask max by catchment
                mask_fl = os.path.join(mask_fl_dir, f'mask_total_{pegel_id}.tif')
                temp_raster = arcpy.sa.ExtractByMask(total_ras, cat_shp)
                temp_raster.save(mask_fl)
                #6 maximum value 
                fl = arcpy.Raster(mask_fl)
                arcpy.CalculateStatistics_management(fl)
                thres = int(math.floor(fl.maximum)) - 1
                #7 reclass raster 
                reclass_fl =  os.path.join(reclass_dir, f'reclass_{pegel_id}.tif')
                total_raster = arcpy.Raster(total_ras)
                temp_recalc = arcpy.sa.Con(total_raster >= thres, 1, 0)
                temp_recalc.save(reclass_fl)
                #8 lfp raster
                lfp_ras = os.path.join(lfp_raster_dir, f'lfp_{pegel_id}.tif')
                temp_reclass = arcpy.sa.Reclassify(reclass_fl, 'Value', arcpy.sa.RemapValue([[0, "NODATA"],
                                                                                             [1, 1],
                                                                                             ["NODATA","NODATA"]]))
                temp_reclass.save(lfp_ras)
                #9 save raster as polyline 
                lfp = os.path.join(lfp_polyline_dir, f'lfp_{pegel_id}.shp')
                arcpy.conversion.RasterToPolyline(lfp_ras, lfp, 'NODATA', '#', 'NO_SIMPLIFY')
                #10 clip lfp by catchment polyline 
                lfp_clip = os.path.join(clip_lfp_polyl_dir, f'lfp_{pegel_id}.shp')
                arcpy.analysis.PairwiseClip(lfp, cat_shp, lfp_clip)

lfp_merit_hydro()

Longest flow path polyline generated from above has an artifact such that polyline is branched, which is not hydrologocially plausible. 
Thus the longest flow path is determined from the polyline using a depth first search.

In [ ]:
def add_fields(path, fields): #helper func 1 
    '''Adds fields in a shapefile/feature class. 
    '''
    for req_field in fields:
        if req_field not in [_.name for _ in arcpy.ListFields(path)]:
            arcpy.management.AddField(path, req_field, 'DOUBLE')

def find_all_paths_undirected(edges, start, ends, path=[]): #helper function 2.1 #helper function nested in 2.
    """
    lfp from lfp_merit_hydro() is a polyline containing different lines. Each line is an edge, and each ends of a line are nodes.
     

    Input:
        i. start: node closest to outlet (merit_hydro drain points used as outlet)
        ii. ends: for a polyline, if a node is used only once and is not start, then it is taken as input node. this calculation is done in get_path_all_attributes()
        iii. Edges: [(starting_node1, ending_node1)]
    Output:
        In case of paths ending at multiple points, this provides nodes in each of the flowpath. 
    """
    path = path + [start]
    
    if start in ends:
        return [path]
    
    # Get all connected nodes (both directions)
    connected_nodes = []
    for edge in edges:
        if edge[0] == start:
            connected_nodes.append(edge[1])
        elif edge[1] == start:  # Check reverse direction
            connected_nodes.append(edge[0])
    
    if not connected_nodes:
        return []
    
    paths = []
    for neighbor in connected_nodes:
        if neighbor not in path:  # Avoid cycles
            new_paths = find_all_paths_undirected(edges, neighbor, ends, path)
            for new_path in new_paths:
                paths.append(new_path)
    return paths

def get_path_all_attributes(lfp_path, snappoint_path, final_lfp_path): #helper function 2. this has another helper function within it described above as 2.1.
    '''This part gives all the attributes required to do a DFS for the lfp. 
    i. length of each edge
    ii. elevation of the node(s) of an edge if needed
    iii. starting point of the flowpath == node closest to the catchment outlet'''
    nodes_from = []
    nodes_to = []
    all_edges = []
    edge_lengths = dict()
    #edge_fids = dict()
    edge_2_outlet_dist = dict() 
    with arcpy.da.SearchCursor(snappoint_path, ['SHAPE@']) as point_cursor:
        for point_row in point_cursor:
            outlet_point = point_row[0]
            with arcpy.da.SearchCursor(lfp_path, ["SHAPE@",'from_node', 'to_node']) as cursor:
                for row in cursor:
                    edge = row[0]
                    
                    #creating edges for dfs
                    from_node, to_node = (row[1], row[2])
                    nodes_from.append(from_node)
                    nodes_to.append(to_node)
                    
                    #create edges
                    all_edges.append((from_node, to_node))
                    
                    #L
                    edge_lengths[(from_node, to_node)] = round(edge.length, 4) #outputs in m
                    
                    #finding the node closest to catchment outlet
                    outlet_distance = edge.distanceTo(outlet_point)
                    edge_2_outlet_dist[outlet_distance] = (from_node, to_node)
    
    #this part separates end nodes from intersection nodes
    nodes_all = nodes_from + nodes_to
    end_nodes = [_ for _ in nodes_all if nodes_all.count(_) == 1]
    int_nodes = list(set([_ for _ in nodes_all if _ not in end_nodes]))
    
    #getting starting node
    starting_edge = edge_2_outlet_dist[min(edge_2_outlet_dist.keys())] #returns tuple of (from,to) of edge closest to outlet
    start_node = list(set(starting_edge).intersection(set(end_nodes)))[0] #tuple and list to sets, and A intersection B.
    
    ends = [_ for _ in end_nodes if _ != start_node]
    
    # print(all_edges, start, ends)
    paths = find_all_paths_undirected(all_edges, start_node, ends, path=[])
    return paths, edge_lengths
    

def find_longest_path(paths, edge_lengths): #helper function 3.
    '''From the paths, this function selects the path with the longest length.
    Inputs:
        i.paths: from find_all_paths_undirected()
        ii. edge_lengths: this is a dictionary containing length of edge e.g. for pathA: {(node1, node2): l1, (node2, node3): l2, (node4, node3): l3}
    Output:
        path with the maximum length. eg [node1, node2, node4]'''
    # print(paths, edge_lengths)
    length = dict()
    for path in paths:
        nodes = [(i,j) for i,j in zip(path[0:-1], path[1:])] + [(i,j) for i,j in zip(path[1:], path[0:-1])]
        path_length = sum([edge_lengths[node] for node in nodes if node in edge_lengths.keys()])
        length[path_length] = path
    max_length = max(length.keys())
    max_path = length[max_length] #dict cant hold more than one key with same value, so no need to deal with duplication
    return max_path

def update_lines_is_lfp(lfp_path, lfp_dfs): #helper function 4. 
    '''This part adds new field to lfp shapefile: is_lfp.
    If an edge in the polyline belongs to the longest flow path [from find_longest_path()], is_lfp =1, else = 0.
    Inputs:
        i. lfp_path = file path for the lfp.
        ii. lfp_dfs = lfp with node ids obtained after dfs
    Output:
        i. Updates the lfp shapefile such that if both nodes of a line in lfp polyline is in lfp_dfs, is_lfp = 1, else 0.
        
    '''
    
    add_fields(lfp_path, ['is_lfp'])
    with arcpy.da.UpdateCursor(lfp_path, ['FID', 'is_lfp', 'from_node', 'to_node']) as cursor:
        for row in cursor:
            edge_fid = row[0]
            if row[-1] in lfp_dfs and row[-2] in lfp_dfs:
                row[1] = 1
            else:
                row[1] = 0
            cursor.updateRow(row)

    
def create_single_lfp(lfp_path, lfp_final_path): #helper function 5. 
    '''from the lfp created from lfp_merit_hydro() and updated by update_lines_is_lfp(), 
        the longest line is selected and dissolved into one line.'''
    where_clause = "is_lfp >= 1"
    temp_layer = r"temp_lfp"
    arcpy.management.MakeFeatureLayer(lfp_path, temp_layer, where_clause) #this selects the lines from polyline
    arcpy.management.Dissolve(temp_layer, lfp_final_path) #this merges those polylines into one line

From the lfp, time of concentration is calculated using Horton's equation. 

In [ ]:
def get_elevation_end_points(final_lfp_path, cat_z_file_path, 
                             lfp_end_point_save_path, lfp_end_z_save_path): #6b
    '''This extracts the elevation of the end points of the longest flow path obtained.
    The dem used for extracting elevation for Bavaria is the raw DEM from Merit Hydro.'''
    arcpy.management.FeatureVerticesToPoints(final_lfp_path, lfp_end_point_save_path, 'BOTH_ENDS')
    zs = []
    with arcpy.da.SearchCursor(lfp_end_point_save_path, ['SHAPE@XY']) as z_cursor:
        for z_row in z_cursor:
            point = z_row[0]
            print(type(point))
            z_point = f'{point[0]} {point[1]}'
            z_res = arcpy.management.GetCellValue(cat_z_file_path, z_point)
            z = float(z_res.getOutput(0).replace(',','.'))
            zs.append(z)
    z_div = max(zs)
    z_outlet = min(zs)
    return z_outlet, z_div
    
def update_single_lfp(final_lfp_path, 
                      pegel_id, 
                      cat_slope_file_path,
                      cat_z_file_path,
                     lfp_slope_save_path,
                     lfp_end_point_save_path, 
                     lfp_end_z_save_path): #helper function 6.
    '''
    1. the dissolved lfp has no attributes. 
    this part adds pegel id, length, z_at end nodes, and average slope along the flow path.
    2. geodesic length of the lfp is calculated. 
    3. using elevations from 6b time of concentration is calculated.
    4. A redundant field for time of concentration from slope exists. Not removed to maintain indexing.
        This redundant field has values of -999.'''
    add_fields(final_lfp_path, ['length_m', 'z1_m', 'z2_m', 'S_perc', 'tc_dz', 'tc_S', 'Pegel_ID'])
    arcpy.management.CalculateGeometryAttributes(final_lfp_path, [['length_m','LENGTH_GEODESIC']], 'METERS')
    
    #check without catchment average slope
    avrg_slope = -999#round(get_lfp_slope(final_lfp_path, cat_slope_file_path, lfp_slope_save_path),4)
    #avrg_slope = None
    
    
    z_min, z_div = get_elevation_end_points(final_lfp_path, cat_z_file_path, lfp_end_point_save_path, lfp_end_z_save_path)
    with arcpy.da.UpdateCursor(final_lfp_path, ['length_m', 'z1_m', 'z2_m', 'S_perc', 'tc_dz', 'tc_S', 'Pegel_ID']) as cursor:
        for row in cursor:
            L = row[0]
            dZ = abs(z_div-z_min)
            tc_dz = 0.0195 * (L ** 0.77) * (abs(z_div-z_min)/L)**-0.385  #this is the true kirpich time of concentration 
            tc_S = -999 #0.0195 * (L ** 0.77) * (avrg_slope/100)**-0.385 #this is only for check
            row[1] = z_min
            row[2] = z_div
            row[3] = avrg_slope
            row[4] = tc_dz
            row[5] = tc_S
            row[6] = pegel_id
            cursor.updateRow(row)
    #this part might throw error, as variables are used after end of context window 
            #print('L, dZ, avrg_slope, tc_dz, tc_S', L, dZ, avrg_slope, tc_dz, tc_S)
            return L, dZ, avrg_slope, tc_dz, tc_S



def update_tc(catchments_shp, 
              lfp_dir, 
              snappoint_dir, 
              final_lfp_dir,
              cat_slope_file_dir,
              cat_z_file_dir,
              lfp_slope_save_dir,
              lfp_end_point_save_dir,
              lfp_z_raster_save_dir,
              pegel_ids_2_calc):
    '''
    This is the main function. 
    Helper functions are described above in the order of application.
    Inputs:
        i. catchments_shp =  file path of the catchments of Bavaria. 
            Original catchments (without smoothening and simplification) are used here.
        ii. snappoint_dir = dir path of gauge points.
        iii. final_lfp_dir = dir path of lfp from lfp_merit_hydro()
        iv. cat_slope_file_dir = dir path of catchment slopes in %
        v. cat_z_file_dir = dir containing clipped HydroDEM for catchments
        vi. lfp_slope_save_dir = dir to save the slope raster masked by lfp
        vii. lfp_end_point_save_dir = dir to save ending nodes of a lfp as xy points
        viii. lfp_z_raster_save_dir = dir to save the ending xy points with extracted elevations 
        ix. pegel_ids_2_calc = selected catchments to calculate. this can be set to None, and the if statement erased to calculate tc for all catchments.
    '''
    add_fields(catchments_shp, ['length_lfp', 'dz_lfp', 'S_lfp', 'tc_s', 'tc_dZ'])
    with arcpy.da.UpdateCursor(catchments_shp, ['Pegel_ID', 'length_lfp', 'dz_lfp', 'S_lfp', 'tc_dZ','tc_s']) as cursor: #, 
        for row in cursor:
            pegel_id = int(row[0])
            
            lfp_file = f'lfp_{pegel_id}.shp'
            lfp_path = os.path.join(lfp_dir, lfp_file)
            snappoint_path = os.path.join(snappoint_dir, f'gauge_{pegel_id}.shp')
            if os.path.exists(lfp_path) and os.path.exists(snappoint_path) == False:
                print('Save snappoint for ', pegel_id)
                
            elif os.path.exists(lfp_path) and os.path.exists(snappoint_path): #the lfp path and snappoint path automatically excludes catchments from ls
                if pegel_id in pegel_ids_2_calc: #this statement can be deactivated, and rest statements shift+tab to calculate for all catchments
                    print(pegel_id)
                    final_lfp_file = f'lfp_{pegel_id}.shp'
                    final_lfp_path = os.path.join(final_lfp_dir, final_lfp_file)
                    
                    catchment_slope_file = f'mask_slope_{pegel_id}.tif'
                    catchment_slope_file_path = os.path.join(cat_slope_file_dir, catchment_slope_file)
                    catchment_z_file = f'elv_{pegel_id}.tif'
                    catchment_z_file_path = os.path.join(cat_z_file_dir, catchment_z_file)    
                    lfp_slope_save_file = f'lfp_S_{pegel_id}.tif'
                    lfp_slope_save_path = os.path.join(lfp_slope_save_dir, lfp_slope_save_file)
                    lfp_end_point_save_file = f'lfp_end_points_{pegel_id}.shp'
                    lfp_end_point_save_path = os.path.join(lfp_end_point_save_dir, lfp_end_point_save_file)
                    lfp_z_raster_save_file = f'lfp_z_{pegel_id}.tif'
                    lfp_z_raster_save_path = os.path.join(lfp_z_raster_save_dir, lfp_z_raster_save_file)
                    
                    paths, edge_lengths = get_path_all_attributes(lfp_path, snappoint_path, final_lfp_path)
                    longest_path = find_longest_path(paths, edge_lengths)
                    update_lines_is_lfp(lfp_path, longest_path)
                    create_single_lfp(lfp_path, final_lfp_path)
                    row[1], row[2], row[3], row[4], row[5] = update_single_lfp(final_lfp_path, 
                                                                               pegel_id, 
                                                                               catchment_slope_file_path, 
                                                                               catchment_z_file_path, 
                                                                               lfp_slope_save_path,
                                                                               lfp_end_point_save_path,
                                                                               lfp_z_raster_save_path)
                    cursor.updateRow(row)


arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

update_tc(catchments_shp = r'.\catchments', 
          lfp_dir = r'.\0merit_hydro\9 clipped lfp', 
          snappoint_dir = r'.\gauge points',
         final_lfp_dir = r'.\999 lfp final\1 final lfp',
         cat_slope_file_dir = r'.\cat_mask slopes',
         cat_z_file_dir = r'.\clipped dem merit hydro correction',
          lfp_slope_save_dir = r'.\999 lfp final\5 lfp slope',
         lfp_end_point_save_dir = r'.\999 lfp final\2 lfp ends',
          lfp_z_raster_save_dir = r'.\999 lfp final\4 lfp ends z',
         pegel_ids_2_calc = None)


In [ ]:
def write_to_featureclass(shp_dir, fc_path):
    '''
    a, Writes all shapefiles into a feature class. This does not create a geodatabase, and feature dataset. They were done manually.
    This part requires that all required polygon shapefiles are in one directory, and point shape files are in another directory.
    The valid_name() function ensures this in this script. 
    Input arguments:
    shp_dir = directory containing all shapefile of a type
    fc_path = path to the feature class'''
    arcpy.env.workspace = shp_dir
    arcpy.env.overwriteOutput = True
    arcpy.DeleteRows_management(fc_path) #removes rows (records) added to the featureclass from previous iteration
    print('Removed old files')
    shapefiles = arcpy.ListFeatureClasses("*.shp")
    for shapefile in shapefiles:
        try:
            arcpy.Append_management(shapefile, fc_path, "NO_TEST")
            #print(f"Appended {shapefile} to {appended_fc}")
        except arcpy.ExecuteError as e:
            print(f"Error appending {shapefile}: {e}")
    print("All shapefiles appended successfully.")
write_to_featureclass(r'.\999 lfp final\1 final lfp',
                     r'.\final.gdb\lfp_merit')